# 03 — Forecasting

Requiere haber ejecutado `01_diagnostico.ipynb` (usa su configuración). No depende de `02_pipeline_nowcast.ipynb` -- forecasting y nowcast son ramas independientes que comparten solo la configuración de variables, no los modelos.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import pickle
import time
from utilidades_tep import construir_features, evaluar_modelo, construir_features_forecast_multi_sim, evaluar_modelo_split_por_simulacion

plt.style.use('ggplot')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Cargar configuración y datos

`construir_features`, `evaluar_modelo` y `construir_features_forecast_multi_sim` vienen del módulo compartido `utilidades_tep.py` (mismo código que usa `02_pipeline_nowcast.ipynb`, sin duplicar) -- así este notebook no depende de haber ejecutado `02_pipeline_nowcast.ipynb` antes, pero tampoco arrastra una copia separada de esas funciones que podría desincronizarse si se corrige un bug en una sola de las dos.

In [ ]:
with open('../artefactos/configuracion.pkl', 'rb') as f:
    configuracion = pickle.load(f)

VARIABLE_CALIDAD = configuracion['VARIABLE_CALIDAD']
VARIABLES_XMV = configuracion['VARIABLES_XMV']
VARIABLES_XMEAS = configuracion['VARIABLES_XMEAS']
redundant_pairs = configuracion['redundant_pairs']
RUTA_CSV = configuracion['RUTA_CSV']
COLUMNA_TIEMPO = configuracion['COLUMNA_TIEMPO']

df_raw = pd.read_csv(RUTA_CSV)
df_raw = df_raw.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)
df_sim = df_raw[df_raw['simulationRun'] == 1].reset_index(drop=True) if 'simulationRun' in df_raw.columns else df_raw

variables_entrada_calidad = VARIABLES_XMV + VARIABLES_XMEAS

print("Configuración cargada. VARIABLE_CALIDAD:", VARIABLE_CALIDAD)

In [ ]:
def _lags_potencia(n_lags_max, base=2):
    lags = []
    l = 1
    while l < n_lags_max:
        lags.append(l)
        l *= base
    lags.append(n_lags_max)
    return sorted(set(lags))


def construir_features(df, variables_entrada, usar_lags=True, n_lags=3,
                        espaciado_lags='consecutivo', base_potencia=2,
                        usar_rolling_mean=False, ventana_mean=5,
                        usar_rolling_std=False, ventana_std=5,
                        usar_diff=False):
    df_out = df.copy()
    columnas_features = list(variables_entrada)

    if espaciado_lags == 'potencia':
        lista_lags = _lags_potencia(n_lags, base=base_potencia)
    else:
        lista_lags = list(range(1, n_lags + 1))

    # Acumulamos las columnas nuevas en un diccionario y las unimos con UN solo concat al
    # final, en vez de insertarlas una a una con df_out[col] = ... . Insertar columna a
    # columna fragmenta el DataFrame internamente (pandas avisa con PerformanceWarning) y
    # es notablemente más lento cuando esta función se llama muchas veces seguidas -- como
    # ocurre en construir_features_multi_sim, que la invoca una vez POR CADA simulación.
    nuevas_columnas = {}
    for var in variables_entrada:
        if usar_lags:
            for lag in lista_lags:
                col = f'{var}_lag_{lag}'
                nuevas_columnas[col] = df_out[var].shift(lag)
                columnas_features.append(col)
        if usar_rolling_mean:
            col = f'{var}_roll_mean_{ventana_mean}'
            nuevas_columnas[col] = df_out[var].shift(1).rolling(window=ventana_mean).mean()
            columnas_features.append(col)
        if usar_rolling_std:
            col = f'{var}_roll_std_{ventana_std}'
            nuevas_columnas[col] = df_out[var].shift(1).rolling(window=ventana_std).std()
            columnas_features.append(col)
        if usar_diff:
            col = f'{var}_diff'
            nuevas_columnas[col] = df_out[var].diff()
            columnas_features.append(col)

    df_out = pd.concat([df_out, pd.DataFrame(nuevas_columnas, index=df_out.index)], axis=1)

    return df_out, columnas_features


def evaluar_modelo(X, y, train_frac=0.7, n_estimators=150, max_depth=3, learning_rate=0.1):
    corte = int(len(X) * train_frac)
    X_train, X_test = X.iloc[:corte], X.iloc[corte:]
    y_train, y_test = y.iloc[:corte], y.iloc[corte:]

    modelo = xgb.XGBRegressor(
        n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
        random_state=42, n_jobs=-1
    )
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return {'modelo': modelo, 'rmse': rmse, 'X_test': X_test, 'y_test': y_test,
            'preds': preds, 'corte': corte}

## 7. Forecasting de la calidad — qué va a pasar si no tocas nada

Hasta ahora, `recomendar_ajuste` parte siempre del **último estado observado** del proceso. Eso tiene una limitación: si el proceso ya está en movimiento hacia un estado peor (por ejemplo, la temperatura llevaba varios pasos subiendo), recomendar sobre el estado *actual* ignora hacia dónde se dirige el proceso *ya*, con o sin tu intervención.

La idea aquí es simple y deliberadamente separada del control (todavía no tocamos XMV): entrenamos un modelo que predice `VARIABLE_CALIDAD` a un horizonte `h` pasos vista, **dejando las XMV fijas en su último valor observado** — es decir, "¿qué va a pasar si el operador no cambia nada?". Ese es el mismo patrón de forecasting que usamos en el notebook de la bomba, aplicado aquí a la variable de calidad del TEP.

**Importante — qué es y qué no es esto:** esto NO es todavía control predictivo (MPC). No estamos simulando "qué pasaría si cambio la XMV X" — solo estamos prediciendo la inercia natural del proceso. El resultado de esta sección se va a usar más adelante como **punto de partida alternativo** para la búsqueda de recomendación, en vez del último dato observado. El salto a "qué pasaría si cambio la XMV" (control predictivo real) es un notebook aparte, porque requiere encadenar predicciones de forma recursiva y ahí el error se acumula paso a paso — no es una extensión trivial de esto.

In [ ]:
HORIZONTE_FORECAST = 5  # nº de muestras hacia adelante (con 'sample' cada 3 min, 5 = 15 minutos vista)

# Mismas features de entrada que el modelo de nowcast -- la única diferencia es el target,
# que aquí desplazamos h pasos hacia el futuro en vez de usar el valor en el mismo instante.
df_feat_fc, cols_feat_fc = construir_features(
    df_sim, variables_entrada_calidad,
    usar_lags=True, n_lags=1, espaciado_lags='consecutivo',
    usar_rolling_mean=False, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat_fc[f'target_futuro_{VARIABLE_CALIDAD}'] = df_sim[VARIABLE_CALIDAD].shift(-HORIZONTE_FORECAST).values
# Guardamos también el valor ACTUAL (sin desplazar) en la misma construcción, para que el
# dropna() de abajo lo arrastre con exactamente los mismos índices que el resto de columnas
# -- así el baseline naive de la siguiente celda queda perfectamente alineado por diseño,
# sin tener que reconstruir el recorte de índices a mano (que es donde es fácil desalinearse).
df_feat_fc['valor_actual_calidad'] = df_sim[VARIABLE_CALIDAD].values
df_feat_fc = df_feat_fc.dropna().reset_index(drop=True)

X_forecast = df_feat_fc[cols_feat_fc]
y_forecast = df_feat_fc[f'target_futuro_{VARIABLE_CALIDAD}']

res_forecast = evaluar_modelo(X_forecast, y_forecast, n_estimators=100, max_depth=6)
print(f"RMSE forecasting {VARIABLE_CALIDAD} a +{HORIZONTE_FORECAST} pasos: {res_forecast['rmse']:.4f}")

# El RMSE de nowcast viene de 02_pipeline_nowcast.ipynb -- si ya lo ejecutaste, lo cargamos
# para comparar; si no, lo omitimos sin que falle este notebook (forecasting es independiente).
try:
    with open('../artefactos/rmse_nowcast.pkl', 'rb') as f:
        rmse_nowcast_guardado = pickle.load(f)
    print(f"Para comparar -- RMSE de nowcast (horizonte 0, de 02_pipeline_nowcast.ipynb): {rmse_nowcast_guardado['rmse_una_sim']:.4f}")
except FileNotFoundError:
    print("(Ejecuta 02_pipeline_nowcast.ipynb si quieres comparar contra el RMSE de nowcast)")
print(f"(Es normal y esperable que el RMSE de forecasting sea mayor: predices algo que aún no ha pasado)")

### ¿RMSE=3.3 con std=5.97 es bueno? -- la comparación correcta no es esa

Comparar el RMSE directamente contra la desviación típica de la señal responde a una pregunta distinta de la que importa. La std mide cuánto varía la señal en **todo** su rango histórico; tu modelo no necesita explicar esa variabilidad completa, necesita predecir mejor que la alternativa más simple posible.

El benchmark correcto para forecasting es un **baseline naive (persistence model)**: predecir que el valor en `t+h` va a ser igual al último valor conocido en `t`. Si tu XGBoost con todo el feature engineering apenas le gana a "no va a cambiar nada", el modelo no está aportando información real más allá de la inercia del proceso que ya conocías por el PACF. Si le gana con margen claro, ahí sí tienes evidencia de que está aprendiendo algo genuino.

In [ ]:
# Baseline naive: predice que el valor en t+h será igual al valor en t (persistence model)
# Se calcula sobre el MISMO conjunto de test que el modelo, para que la comparación sea justa.
# Usamos "valor_actual_calidad", que se guardó DENTRO de df_feat_fc antes del dropna() --
# por construcción tiene exactamente los mismos índices que y_forecast, sin necesidad de
# reconstruir el recorte a mano (que es donde es fácil desalinearse sin que salte ningún error).
corte_fc = res_forecast["corte"]
valor_actual_test = df_feat_fc["valor_actual_calidad"].iloc[corte_fc:].reset_index(drop=True)

y_test_fc = res_forecast["y_test"].reset_index(drop=True)
rmse_naive = np.sqrt(mean_squared_error(y_test_fc, valor_actual_test))
rmse_modelo = res_forecast["rmse"]

print(f"RMSE del modelo XGBoost (forecasting +{HORIZONTE_FORECAST}):  {rmse_modelo:.4f}")
print(f"RMSE del baseline naive (no cambia nada):     {rmse_naive:.4f}")

mejora_pct = (1 - rmse_modelo / rmse_naive) * 100
print(f"\nMejora del modelo sobre el baseline naive: {mejora_pct:+.1f}%")

if mejora_pct < 5:
    print("ALERTA: El modelo apenas mejora sobre no va a cambiar nada -- gran parte de tu RMSE bajo")
    print("    puede deberse a que el proceso es lento/estable en este horizonte, no a que el modelo")
    print("    esté aprendiendo relaciones causales útiles. Revisa si el horizonte es demasiado corto")
    print("    para que el forecasting aporte algo que el propio valor actual no diera ya.")
elif mejora_pct < 20:
    print("El modelo mejora sobre el baseline, pero de forma modesta -- hay margen para revisar")
    print("features, horizonte, o aceptar que el proceso tiene poca variabilidad predecible aquí.")
else:
    print("El modelo mejora claramente sobre el baseline naive -- señal de que está capturando")
    print("  relaciones reales entre las XMV/XMEAS y la calidad futura, no solo inercia.")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(res_forecast['y_test'].values, label='Real', color='blue', alpha=0.6)
plt.plot(res_forecast['preds'], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
plt.title(f"Forecasting a +{HORIZONTE_FORECAST} pasos — RMSE {res_forecast['rmse']:.3f}")
plt.xlabel('Muestras de test')
plt.ylabel(f'{VARIABLE_CALIDAD} (futuro)')
plt.legend()
plt.tight_layout()
plt.show()

## 7bis. Forecasting entrenado con todo el dataset (400 train / 100 test, por simulación)

Mismo criterio que en nowcast (sección 6bis): entrenar con una sola simulación da una estimación optimista y poco fiable, porque el modelo puede estar memorizando las particularidades de esa "historia" concreta en vez de aprender la dinámica general del proceso.

Aquí hay una fuga adicional a la que no basta con aplicar `construir_features_multi_sim` tal cual: el **target** de forecasting (`shift(-h)`, el valor h pasos en el futuro) también tiene que calcularse **dentro de cada simulación por separado**. Si se calcula sobre el dataframe ya concatenado, las últimas `h` filas de cada simulación tomarían como "futuro" datos de la simulación siguiente -- que es un experimento completamente distinto, no una continuación real. Es la misma fuga que ya evitamos para los lags, aplicada ahora al target en vez de a los inputs.

In [ ]:
# simulaciones_train / simulaciones_test vienen de la configuración cargada en la celda
# de configuración de este notebook (mismo split que usa 02_pipeline_nowcast.ipynb)
simulaciones_train = configuracion['simulaciones_train']
simulaciones_test = configuracion['simulaciones_test']

t0 = time.time()
df_feat_fc_multi, cols_feat_fc_multi, nombre_target_fc = construir_features_forecast_multi_sim(
    df_raw, 'simulationRun', variables_entrada_calidad, VARIABLE_CALIDAD, HORIZONTE_FORECAST,
    usar_lags=True, n_lags=2, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=3,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat_fc_multi = df_feat_fc_multi.dropna().reset_index(drop=True)
t1 = time.time()
print(f"Features de forecasting calculadas por simulación: {t1-t0:.2f}s")
print(f"Filas totales tras construir features y dropna: {len(df_feat_fc_multi)}")

t0 = time.time()
res_forecast_multi = evaluar_modelo_split_por_simulacion(
    df_feat_fc_multi, cols_feat_fc_multi, nombre_target_fc, 'simulationRun',
    simulaciones_train, simulaciones_test,
    n_estimators=100, max_depth=5
)
t1 = time.time()

rmse_forecast_multi = res_forecast_multi['rmse']
rmse_forecast_una_sim = res_forecast['rmse']

print(f"\nEntrenamiento + evaluación: {t1-t0:.2f}s")
print(f"RMSE forecasting +{HORIZONTE_FORECAST} con split por simulación completa: {rmse_forecast_multi:.4f}")
print(f"Para comparar -- RMSE forecasting +{HORIZONTE_FORECAST} con solo simulationRun==1: {rmse_forecast_una_sim:.4f}")

### Real vs predicho, sobre las 100 simulaciones de test

Para verlo "en predicción" como pediste: aquí no hay una sola serie continua de test (son 100 simulaciones distintas concatenadas), así que el gráfico de líneas real-vs-predicho tiene sentido por tramos, no como una sola curva continua -- cada simulación de test es independiente de la siguiente. Mostramos primero un scatter (real vs predicho, la forma estándar de evaluar esto cuando el test no es una serie temporal única) y después una muestra de una sola simulación de test a modo de ejemplo visual de la forma.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter real vs predicho -- la forma correcta de ver el ajuste cuando el test
# son 100 simulaciones independientes, no una sola serie continua
axes[0].scatter(res_forecast_multi['y_test'], res_forecast_multi['preds'], alpha=0.15, s=8, color='steelblue')
lims = [res_forecast_multi['y_test'].min(), res_forecast_multi['y_test'].max()]
axes[0].plot(lims, lims, color='crimson', linestyle='--', label='predicción perfecta')
axes[0].set_xlabel('Real')
axes[0].set_ylabel('Predicho')
axes[0].set_title(f'Real vs predicho -- {len(simulaciones_test)} simulaciones de test\nRMSE={rmse_forecast_multi:.4f}')
axes[0].legend()

# Ejemplo visual: una sola simulación de test, para ver la forma temporal de la predicción
sim_ejemplo = simulaciones_test[0]
mask_ejemplo = df_feat_fc_multi.loc[res_forecast_multi['X_test'].index, 'simulationRun'] == sim_ejemplo
axes[1].plot(res_forecast_multi['y_test'][mask_ejemplo].values, label='Real', color='blue', alpha=0.7)
axes[1].plot(res_forecast_multi['preds'][mask_ejemplo], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
axes[1].set_title(f'Ejemplo: simulación {sim_ejemplo} (una de las 100 de test)')
axes[1].set_xlabel('Muestras dentro de esa simulación')
axes[1].legend()

plt.tight_layout()
plt.show()

### Usar el estado proyectado como punto de partida de la recomendación

Con el modelo de forecasting ya entrenado, podemos predecir *ahora* cuál será previsiblemente la calidad dentro de `HORIZONTE_FORECAST` pasos si nadie toca nada. Eso por sí solo ya es información útil para un operador ("si no haces nada, la calidad va a caer"), y además nos da pie a comparar dos formas de recomendar:

- **Recomendación reactiva** (la que ya tenías): busca el mejor ajuste de XMV partiendo del **último estado observado**.
- **Recomendación anticipativa**: busca el mejor ajuste de XMV partiendo del **estado que el proceso tendrá dentro de `h` pasos si no intervienes** — es decir, se adelanta a la deriva natural del proceso en vez de reaccionar solo a lo que ya pasó.

Ojo: seguimos sin simular qué pasaría *con* la intervención a lo largo de esos `h` pasos — solo cambiamos el punto de partida de la búsqueda. Es una mejora real, pero modesta comparada con MPC.

In [ ]:
calidad_actual = df_sim[VARIABLE_CALIDAD].iloc[-1]
calidad_proyectada = res_forecast['modelo'].predict(df_feat_fc[cols_feat_fc].iloc[[-1]])[0]

print(f"Calidad actual (último dato observado):              {calidad_actual:.4f}")
print(f"Calidad proyectada a +{HORIZONTE_FORECAST} pasos (sin intervención): {calidad_proyectada:.4f}")

diferencia = calidad_proyectada - calidad_actual
if abs(diferencia) > res_forecast['rmse']:
    tendencia = "empeorando" if diferencia < 0 else "mejorando"
    print(f"\n⚠️  El proceso parece estar {tendencia} por su cuenta (diferencia mayor que el error típico del modelo).")
    print("   Puede tener sentido recomendar sobre el estado proyectado, no solo sobre el actual.")
else:
    print("\nLa diferencia entre el estado actual y el proyectado es pequeña frente al error del modelo -- el proceso está razonablemente estable, no hay una deriva clara que anticipar.")

## Guardar artefactos para el notebook de recomendación

Guardamos el modelo de forecasting (necesario para la variante "anticipativa" de recomendación) y sus datos de features.

In [ ]:
import joblib

joblib.dump(res_forecast['modelo'], '../artefactos/modelo_forecast_una_sim.joblib')
df_feat_fc.to_csv('../artefactos/df_feat_forecast.csv', index=False)
with open('../artefactos/cols_feat_forecast.pkl', 'wb') as f:
    pickle.dump(cols_feat_fc, f)
with open('../artefactos/proyeccion_calidad.pkl', 'wb') as f:
    pickle.dump({'calidad_actual': calidad_actual, 'calidad_proyectada': calidad_proyectada}, f)

if 'res_forecast_multi' in dir():
    joblib.dump(res_forecast_multi['modelo'], '../artefactos/modelo_forecast_multi_sim.joblib')
    df_feat_fc_multi.to_csv('../artefactos/df_feat_forecast_multi.csv', index=False)
    print("Guardado: modelo_forecast_una_sim.joblib + modelo_forecast_multi_sim.joblib")
else:
    print("Guardado: modelo_forecast_una_sim.joblib")